## 1. Setup

In [1]:
!pip -q install fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 21.6 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import time

import pandas as pd
from fastembed import SparseTextEmbedding

SEED = 42

MODEL_NAME = "Qdrant/bm25"

print("Setup complete.")

Setup complete.


## 2. Inspect Linked dataset

In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "products.parquet":
            print(os.path.join(root, file))

/kaggle/input/notebooks/jndhruv/02-data-cleaning/multi-modal-fashion-ecom/data/processed/products.parquet


In [4]:
PRODUCTS_PATH = Path(
    "/kaggle/input/notebooks/jndhruv/02-data-cleaning/"
    "multi-modal-fashion-ecom/data/processed/products.parquet"
)

products_df = pd.read_parquet(PRODUCTS_PATH)

print("Products:", len(products_df))
print("Columns:", products_df.columns.tolist())

Products: 44419
Columns: ['id', 'product_display_name', 'brand_name', 'gender', 'master_category', 'sub_category', 'article_type', 'base_colour', 'season', 'usage', 'year', 'price', 'discounted_price', 'description', 'image_url', 'pattern', 'fabric', 'sleeve_length', 'occasion', 'fit', 'neck', 'length', 'search_text']


In [5]:
assert len(products_df) == 44_419
assert products_df["id"].is_unique
assert products_df["search_text"].notna().all()

print("Product count:", len(products_df))
print("Unique IDs:", products_df["id"].nunique())
print("Missing search_text:", products_df["search_text"].isna().sum())

print("\nCatalogue validation passed.")

Product count: 44419
Unique IDs: 44419
Missing search_text: 0

Catalogue validation passed.


## 3. Load Qdrant/Bm25 Model

In [6]:
MODEL_NAME = "Qdrant/bm25"

sparse_model = SparseTextEmbedding(
    model_name=MODEL_NAME
)

print("Sparse model:", MODEL_NAME)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Sparse model: Qdrant/bm25


## 4. Sparse Embedding Smoke Test

In [7]:
sample_texts = products_df["search_text"].head(5).tolist()
sample_ids = products_df["id"].head(5).tolist()

sample_embeddings = list(
    sparse_model.embed(sample_texts)
)

print("Number of embeddings:", len(sample_embeddings))

for pid, embedding in zip(sample_ids, sample_embeddings):
    print(
        f"ID: {pid} | "
        f"non-zero terms: {len(embedding.indices)}"
    )

Number of embeddings: 5
ID: 1566 | non-zero terms: 7
ID: 39342 | non-zero terms: 54
ID: 13182 | non-zero terms: 90
ID: 21762 | non-zero terms: 59
ID: 48692 | non-zero terms: 22


In [8]:
for embedding in sample_embeddings:
    assert len(embedding.indices) == len(embedding.values)

print("Sparse vector structure validated.")

Sparse vector structure validated.


## 5. Benchmarking Generation

In [9]:
BENCHMARK_SIZE = 1_000

benchmark_texts = (
    products_df["search_text"]
    .head(BENCHMARK_SIZE)
    .tolist()
)

start_time = time.perf_counter()

benchmark_embeddings = list(
    sparse_model.embed(benchmark_texts)
)

elapsed = time.perf_counter() - start_time

print(f"Texts processed: {len(benchmark_embeddings)}")
print(f"Elapsed time: {elapsed:.2f} seconds")
print(f"Throughput: {len(benchmark_embeddings) / elapsed:.2f} texts/s")

Texts processed: 1000
Elapsed time: 0.31 seconds
Throughput: 3222.51 texts/s


In [10]:
estimated_seconds = (
    44_419 /
    (len(benchmark_embeddings) / elapsed)
)

print(
    f"Estimated full generation time: "
    f"{estimated_seconds / 60:.2f} minutes"
)

Estimated full generation time: 0.23 minutes


## 6. Generate Full Sparse Vectors

In [11]:
start_time = time.perf_counter()

sparse_vectors = {}

texts = products_df["search_text"].tolist()
ids = products_df["id"].tolist()

for pid, embedding in zip(
    ids,
    sparse_model.embed(texts)
):
    sparse_vectors[int(pid)] = {
        "indices": embedding.indices.tolist(),
        "values": embedding.values.tolist(),
    }

elapsed = time.perf_counter() - start_time

print("Generation complete.")
print("Products:", len(sparse_vectors))
print(f"Elapsed time: {elapsed:.2f} seconds")
print(
    f"Throughput: "
    f"{len(sparse_vectors) / elapsed:.2f} texts/s"
)

Generation complete.
Products: 44419
Elapsed time: 13.66 seconds
Throughput: 3251.61 texts/s


## 7. Validation

In [12]:
assert len(sparse_vectors) == 44_419

product_ids = set(
    products_df["id"].astype(int)
)

assert set(sparse_vectors.keys()) == product_ids

for pid, vector in sparse_vectors.items():
    assert "indices" in vector
    assert "values" in vector
    assert len(vector["indices"]) == len(vector["values"])

print("Sparse vector artifact validated.")
print("Products:", len(sparse_vectors))

Sparse vector artifact validated.
Products: 44419


In [13]:
nnz = [
    len(vector["indices"])
    for vector in sparse_vectors.values()
]

print("Non-zero terms:")
print("Min:", min(nnz))
print("Max:", max(nnz))
print("Mean:", sum(nnz) / len(nnz))

Non-zero terms:
Min: 6
Max: 139
Mean: 57.26835363245458


## 8. Output

In [14]:
OUTPUT_PATH = Path(
    "/kaggle/working/sparse_vectors.json"
)

with open(OUTPUT_PATH, "w") as f:
    json.dump(sparse_vectors, f)

print("Saved:", OUTPUT_PATH)

Saved: /kaggle/working/sparse_vectors.json


In [15]:
with open(OUTPUT_PATH) as f:
    loaded_sparse_vectors = json.load(f)

assert len(loaded_sparse_vectors) == 44_419

print(
    "Reload verification passed:",
    len(loaded_sparse_vectors)
)

Reload verification passed: 44419
